# 04 — REVE (foundation model EEG)
`brain-bzh/reve-large` — transformer pre-addestrato su >60k ore di EEG, con positional encoding 4D adattivo a montaggi variabili. **Input a 200 Hz** (i dati vengono resamplati 256→200 automaticamente) + posizioni elettrodi da `brain-bzh/reve-positions`. Preprocessing = `PP_MINIMAL` (default).

Qui REVE è usato come **feature extractor congelato + testa lineare**, oppure con **fine-tuning** (`freeze_backbone=False`).

**Perché REVE**: è l'unico modello da cui aspettarsi qualcosa sul **cross-subject (subject-independent)**, dove tutto il resto (ConvNet e grafi) è a chance. Il pretraining su 60k ore potrebbe dare feature che generalizzano tra soggetti.

**Requisiti**: `pip install transformers` e connessione internet al primo run per scaricare i pesi.

> ⚠️ L'API esatta di REVE va verificata al primo run: `REVEClassifier` gestisce in modo difensivo l'output (tensore / `last_hidden_state` / `pooler_output`) e **costruisce la testa lineare subito in `__init__`** (via forward fittizio) — così i suoi pesi entrano nell'optimizer. Se il forward fallisce, incolla l'errore per adeguare `_extract` alla firma reale di `model(eeg, positions)`.

In [ ]:
# --- setup: rende importabili i moduli track3_*.py ---
import sys, os
sys.path.insert(0, os.path.abspath('.'))
import numpy as np, matplotlib.pyplot as plt
import track3_config as C, track3_io as io, track3_preproc as P
print(C.summary())
assert C.DATA_ROOT is not None, C._no_data_msg()
import track3_train as T
import track3_models as M
device=C.get_device(); print('device:', device)

## 1. Test rapido di caricamento (1 soggetto, poche epoche)
Scarica i pesi la prima volta. Se qui va, il run completo è solo questione di tempo.

In [ ]:
df_reve, res_reve = T.run_subject_dependent(
    'reve', subjects=[1], model_kwargs=dict(freeze_backbone=True),
    train_kwargs=dict(epochs=30, patience=10, lr=1e-3, batch_size=16))
df_reve

## 2. Run completo sui 3 protocolli
Scommenta quando il test sopra è andato a buon fine. Il **subject-independent** è il più interessante.

In [ ]:
# --- subject-dependent (15 modelli) ---
# df_d, res_d = T.run_subject_dependent('reve', model_kwargs=dict(freeze_backbone=True),
#     train_kwargs=dict(epochs=50, patience=15, lr=1e-3, batch_size=16))
# T.save_metrics(df_d, 'reve_dependent'); T.plot_per_subject(df_d, 'reve'); plt.show()

# --- subject-mixed (1 modello) ---
# df_m, res_m = T.run_subject_mixed('reve', model_kwargs=dict(freeze_backbone=True),
#     train_kwargs=dict(epochs=50, patience=15, lr=1e-3, batch_size=16))

# --- subject-independent (l'angolo chiave per REVE: soggetti di test mai visti) ---
# df_i, res_i = T.run_subject_independent('reve', mode='holdout', model_kwargs=dict(freeze_backbone=True),
#     train_kwargs=dict(epochs=50, patience=15, lr=1e-3, batch_size=16))
# print('REVE independent (holdout):', df_i.iloc[0]['test_acc'])

## 3. Fine-tuning (se il congelato rende poco)
Se REVE congelato + testa lineare dà poco, sblocca il backbone (`freeze_backbone=False`) con LR più basso.
Più lento (aggiorna 60k-ore di pesi) ma più potente. Consigliato batch piccolo.

```python
df, res = T.run_subject_independent('reve', mode='holdout',
    model_kwargs=dict(freeze_backbone=False),
    train_kwargs=dict(epochs=40, patience=12, lr=1e-4, weight_decay=1e-2, batch_size=8))
```